# Batch CSV time-offset correction

As cropping of the LLS data sets retained only the global time steps, such as 0 min, 5 min, and 10 min, but not the acquisition-time differences between individual tiles, a manual time correction was applied to the time columns of the extracted surface outputs. Applied time offsets were calculated from the acquisition time at the center of each approximately 250 µm × 250 µm tile region. 

This notebook copies CSV files from an input folder into an output folder while correcting the time column.

It is designed to handle Imaris-style CSV exports that may contain preamble lines before the real table header, for example:

```text
Surface
 ====================
Volume [µm^3],Area [µm^2],...,Time [s],Time Index,...
```

Detected time columns include:

- `Time [s]`
- `Time[s]`
- `Time [min]`
- `Time[min]`

Offsets are entered in **seconds**. If the detected column is minutes, the offset is automatically converted to minutes before being added.

## Filename logic

Expected filename pattern:

`Date-of-Measurement_runX_condition_tile-measurement.csv`

Examples:

- `260417_run1_Durva_2-3.csv` → offset key `Durva_2`
- `260320_run2_Enva_1-2.csv` → offset key `Enva_1`
- `260320_run2_Nivo_2-2.csv` → offset key `Nivo_2`

The script uses the condition name plus the first number of the final tile-measurement code.

In [ ]:
from pathlib import Path
import pandas as pd
import re
import csv

# =========================
# USER SETTINGS
# =========================

# Folder containing the original CSV files.
# Use r"..." for Windows paths.
INPUT_FOLDER = Path(r"**** \SampleData\Individual-run_TestData_Script1-3\run1_ImarisSurfaces")

# Output folder. It will be created automatically.
OUTPUT_FOLDER = INPUT_FOLDER / "Time_offset_corrected"

# Separator setting:
# None = auto-detect comma, semicolon, or tab from the real header row
# ","  = force comma-separated CSV
# ";"  = force semicolon-separated CSV
# "\t" = force tab-separated file
CSV_SEPARATOR = None

# Preserve preamble lines before the real CSV header.
# For Imaris exports, this keeps lines like:
#   Surface
#    ====================
PRESERVE_PREAMBLE = True

# Output filename behavior:
# True  -> 260417_run1_Durva_2-3_timeOffsetCorrected.csv
# False -> 260417_run1_Durva_2-3.csv saved inside OUTPUT_FOLDER
ADD_SUFFIX_TO_OUTPUT_FILES = False

# Manual offsets in SECONDS.
# Key = condition + "_" + first number of the final tile-measurement code.
# Example:
#   260417_run1_Durva_2-3.csv -> key "Durva_2"
OFFSETS_SECONDS = {
    "Durva_1": 28.7,
    #"Durva_2": 97.5,
    #"Enva_1": 67.5,
    #"Enva_2": 82.5,
    "aPDL_1": 20.4,
    #"aPDL_2": 44,
    "aPD_1": 12.4,
    #"aPD_2": 27.8,
    #"mw11h317_1": 97.5,
    #"mw11h317_2": 112.5,
    #"Nivo_1": 6.5,
    #"Nivo_2": 19,
    "mock_1": 4.1,
    #"mock_2": 11.1,

    # Add or change more entries here as needed, for example:
    # "Nivo_1": 45,
    # "Nivo_2": 50,
    # "mock_1": 0,
    # "mock_2": 0,
}

## Helper functions

In [ ]:
def parse_condition_and_tile_group(csv_path: Path):
    """
    Parse filenames like:
        260417_run1_Durva_2-3.csv
        260320_run2_mw11h317_1-2.csv

    Returns:
        condition: "Durva"
        tile_group: "2"
        offset_key: "Durva_2"
    """
    stem = csv_path.stem
    parts = stem.split("_")

    if len(parts) < 4:
        raise ValueError(
            f"Filename does not match expected pattern: {csv_path.name}. "
            "Expected something like Date_runX_condition_tile-measurement.csv"
        )

    condition = parts[-2]
    tile_measurement = parts[-1]

    match = re.match(r"^(\d+)-(\d+)$", tile_measurement)
    if not match:
        raise ValueError(
            f"Could not parse tile-measurement code in {csv_path.name}. "
            f"Expected final part like 1-1, 1-2, 2-1, etc., got: {tile_measurement}"
        )

    tile_group = match.group(1)
    offset_key = f"{condition}_{tile_group}"

    return condition, tile_group, offset_key


def detect_separator_from_line(line: str):
    """
    Detect likely separator from a candidate header row.
    Returns None if the line does not look like a table row.
    """
    candidates = [",", ";", "\t"]
    counts = {sep: line.count(sep) for sep in candidates}
    best_sep = max(counts, key=counts.get)

    if counts[best_sep] == 0:
        return None

    return best_sep


def split_header_line(line: str, sep: str):
    """
    Split a header row using the csv module, so quoted fields are handled correctly.
    """
    return next(csv.reader([line], delimiter=sep))


def is_time_column_name(column_name: str):
    """
    Accepts:
        Time [s]
        Time[s]
        Time [sec]
        Time [seconds]
        Time [min]
        Time[min]
        Time [minutes]

    Rejects:
        Time
        Time Index
    """
    cleaned = str(column_name).strip().lower()
    cleaned = re.sub(r"\s+", "", cleaned)

    seconds_names = {"time[s]", "time[sec]", "time[second]", "time[seconds]"}
    minutes_names = {"time[min]", "time[mins]", "time[minute]", "time[minutes]"}

    if cleaned in seconds_names:
        return "seconds"
    if cleaned in minutes_names:
        return "minutes"

    return None


def find_header_row_and_time_column(csv_path: Path, forced_sep=None):
    """
    Finds the real CSV header row and the correct time column.

    This avoids the problem where Imaris preamble lines make pandas
    interpret 'Surface' or another metadata line as the header.
    """
    with open(csv_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        all_lines = f.readlines()

    for row_index, line in enumerate(all_lines):
        if not line.strip():
            continue

        sep = forced_sep if forced_sep is not None else detect_separator_from_line(line)

        # Metadata lines like "Surface" have no separator.
        if sep is None:
            continue

        # Only consider lines with at least a few columns.
        # This avoids short metadata lines.
        if line.count(sep) < 2:
            continue

        columns = split_header_line(line, sep)

        for col in columns:
            unit = is_time_column_name(col)
            if unit is not None:
                return row_index, sep, col, unit, all_lines[:row_index]

    raise ValueError(
        "Could not find a valid time column. Expected one of: "
        "'Time [s]', 'Time[s]', 'Time [min]', or 'Time[min]'."
    )


def read_imaris_or_plain_csv(csv_path: Path, forced_sep=None):
    """
    Reads either a plain CSV or an Imaris-style CSV with preamble lines.

    Returns:
        df
        header_row_index
        detected_separator
        time_column
        time_unit
        preamble_lines
    """
    header_row_index, sep, time_column, time_unit, preamble_lines = find_header_row_and_time_column(
        csv_path,
        forced_sep=forced_sep,
    )

    df = pd.read_csv(
        csv_path,
        sep=sep,
        skiprows=header_row_index,
        header=0,
        engine="python",
    )

    return df, header_row_index, sep, time_column, time_unit, preamble_lines


def make_output_name(csv_path: Path):
    if ADD_SUFFIX_TO_OUTPUT_FILES:
        return f"{csv_path.stem}_timeOffsetCorrected.csv"
    return csv_path.name


def save_corrected_csv(df: pd.DataFrame, output_path: Path, sep: str, preamble_lines=None):
    """
    Save corrected data.

    If PRESERVE_PREAMBLE is True, the original preamble lines are written before
    the corrected table. Otherwise, only the clean table is written.
    """
    if PRESERVE_PREAMBLE and preamble_lines:
        with open(output_path, "w", encoding="utf-8", newline="") as f:
            f.writelines(preamble_lines)
        df.to_csv(output_path, index=False, sep=sep, mode="a")
    else:
        df.to_csv(output_path, index=False, sep=sep)

## Process all CSV files

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

summary_rows = []

csv_files = sorted(INPUT_FOLDER.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {INPUT_FOLDER}")

for csv_path in csv_files:
    try:
        condition, tile_group, offset_key = parse_condition_and_tile_group(csv_path)

        if offset_key not in OFFSETS_SECONDS:
            summary_rows.append({
                "file": csv_path.name,
                "status": "skipped",
                "reason": f"No offset configured for key '{offset_key}'",
                "condition": condition,
                "tile_group": tile_group,
                "offset_key": offset_key,
                "offset_seconds": None,
                "time_column": None,
                "time_unit": None,
                "header_row_index_zero_based": None,
                "detected_separator": None,
                "output_file": None,
            })
            continue

        df, header_row, sep, time_column, time_unit, preamble_lines = read_imaris_or_plain_csv(
            csv_path,
            forced_sep=CSV_SEPARATOR,
        )

        corrected = df.copy()

        if time_unit == "seconds":
            offset_to_add = OFFSETS_SECONDS[offset_key]
        elif time_unit == "minutes":
            offset_to_add = OFFSETS_SECONDS[offset_key] / 60.0
        else:
            raise ValueError(f"Unknown time unit detected for column {time_column}: {time_unit}")

        corrected[time_column] = pd.to_numeric(corrected[time_column], errors="raise") + offset_to_add

        output_name = make_output_name(csv_path)
        output_path = OUTPUT_FOLDER / output_name

        save_corrected_csv(
            corrected,
            output_path=output_path,
            sep=sep,
            preamble_lines=preamble_lines,
        )

        summary_rows.append({
            "file": csv_path.name,
            "status": "processed",
            "reason": "",
            "condition": condition,
            "tile_group": tile_group,
            "offset_key": offset_key,
            "offset_seconds": OFFSETS_SECONDS[offset_key],
            "time_column": time_column,
            "time_unit": time_unit,
            "header_row_index_zero_based": header_row,
            "detected_separator": repr(sep),
            "output_file": output_path.name,
        })

    except Exception as exc:
        summary_rows.append({
            "file": csv_path.name,
            "status": "error",
            "reason": str(exc),
            "condition": None,
            "tile_group": None,
            "offset_key": None,
            "offset_seconds": None,
            "time_column": None,
            "time_unit": None,
            "header_row_index_zero_based": None,
            "detected_separator": None,
            "output_file": None,
        })

summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_FOLDER / "time_offset_correction_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Done. Processed {sum(summary['status'] == 'processed')} file(s).")
print(f"Skipped {sum(summary['status'] == 'skipped')} file(s).")
print(f"Errors {sum(summary['status'] == 'error')} file(s).")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Summary file: {summary_path}")

display(summary)

## Optional sanity check

Run this to compare the first few original and corrected time values for one processed file.

In [ ]:
processed = summary[summary["status"] == "processed"]

if processed.empty:
    print("No processed files to preview.")
else:
    row = processed.iloc[0]
    original_path = INPUT_FOLDER / row["file"]
    corrected_path = OUTPUT_FOLDER / row["output_file"]

    original_df, *_original_info = read_imaris_or_plain_csv(original_path, forced_sep=CSV_SEPARATOR)
    corrected_df, *_corrected_info = read_imaris_or_plain_csv(corrected_path, forced_sep=CSV_SEPARATOR)

    time_col = row["time_column"]

    preview = pd.DataFrame({
        "original_time": original_df[time_col].head(10),
        "corrected_time": corrected_df[time_col].head(10),
        "difference": corrected_df[time_col].head(10) - original_df[time_col].head(10),
    })

    print(f"Preview file: {row['file']}")
    print(f"Time column: {time_col}")
    print(f"Applied offset: {row['offset_seconds']} seconds")
    display(preview)